# Intro to Machine Learning
- Following the course on Kaggle

## Lesson 1: How Models Work (Core Intuition)

### 1. The Core Idea of Machine Learning
Machine learning s a subset of artificial intelligence where computers use algorithms to identify patterns in data and make predictions or decisions, rather than following hard-coded instructions
* **The Process:** Instead of using hard-coded rules, we feed a model data, and it discovers the relationships automatically.

### 2. The Basic Building Block: The Decision Tree
While advanced architectures exist, the **Decision Tree** is the ideal starting point because it is intuitive, highly visual, and forms the foundation for powerful ensemble methods (like Random Forests).

* **How a Simple Tree Works:** * It splits data into categories based on a specific feature (e.g., *Has > 2 bedrooms?* $\rightarrow$ **Yes** / **No**).
  * The prediction at the bottom of the tree is the **historical average** of the data points that fell into that specific category.

### 3. Key Terminology
* **Fitting / Training:** The process of capturing patterns from the data to build the model's structure.
* **Training Data:** The historical dataset used to fit the model.
* **Splits:** Points where the tree divides the data based on a specific characteristic or feature.
* **Tree Depth:** A measure of how many splits the tree makes before reaching a prediction. Deeper trees capture more complex, multi-factor relationships (e.g., factoring in bedrooms, lot size, *and* location).
* **Leaf:** The final point at the bottom of the tree where no further splits occur and a prediction is made.

### 4. Continuous Improvement
* **Shallow Trees (1-2 splits):** Risk failing to capture important factors affecting the target variable (e.g., predicting house price based *only* on the number of bedrooms ignores square footage, location, etc.).
* **Deeper Trees:** Allow the model to capture nuances by tracing a specific path through multiple characteristics to arrive at a highly tailored leaf prediction.

## Lesson 3: Your First Machine Learning Model

### 1. Selecting Data for Modeling
Datasets often contain more variables than we need or can easily interpret. The first step in building a model is paring down the data to relevant components using your intuition.
* To view all available columns in a Pandas DataFrame, use the `.columns` property:
  ```python
  melbourne_data.columns
  ```
* *A feature is an individual, measurable piece of data or characteristic used by a machine learning model to make a prediction or classification

### 2. Standard Variables: Target vs. Features
* By convention, machine learning workflows split the data into two primary pieces: The Prediction Target ($y$): The single variable you want the model to predict.
    * Select this using dot notation (e.g., y = melbourne_data.Price).
    * Stored as a Pandas Series (a single column of data).
* Features ($X$): The columns inputted into the model to determine the prediction target.
    * Select multiple features by providing a list of column names inside brackets (e.g., X = melbourne_data[feature_list]).
    * Stored as a Pandas DataFrame (multiple columns).

### 3. The 4 Steps to Building a Model (scikit-learn)
The Scikit-Learn library (written as sklearn in code) is the industry standard for modeling tabular data. Every model you build follows the same distinct lifecycle:
1. Define: Choose the type of model (e.g., a Decision Tree) and specify its parameters.
2. Fit: Capture patterns from the provided features ($X$) and target ($y$). This is the core training step.
3. Predict: Apply the fitted patterns to make predictions on new or existing data.
4. Evaluate: Determine the numerical accuracy of the model's predictions.

In [21]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

# Get data
melbourne_file_path = '../data/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path)
# display(melbourne_data.describe())

# Drop missing values to keep the initial model simple
melbourne_data = melbourne_data.dropna(axis=0)

# Define the prediction target
y = melbourne_data.Price

# Define the features
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longitude']
X = melbourne_data[melbourne_features]

# Define model
melbourne_model = DecisionTreeRegressor(random_state=1)
# Note on random_state: Many machine learning models allow some internal randomness during training. Setting a static random_state=1 enforces reproducibility, ensuring your code outputs the exact same results every time it is run.

# Fit Model
melbourne_model.fit(X, y)

# Predict
print("Making predictions for the following 5 houses:")
print(X.head())

print("\n The predictions are:")
print(melbourne_model.predict(X.head()))

Making predictions for the following 5 houses:
   Rooms  Bathroom  Landsize  BuildingArea  YearBuilt  Lattitude  Longitude
1      2       1.0     156.0          79.0     1900.0   -37.8079   144.9934
2      3       2.0     134.0         150.0     1900.0   -37.8093   144.9944
4      4       1.0     120.0         142.0     2014.0   -37.8072   144.9941
6      3       2.0     245.0         210.0     1910.0   -37.8024   144.9993
7      2       1.0     256.0         107.0     1890.0   -37.8060   144.9954

 The predictions are:
[1035000. 1465000. 1600000. 1876000. 1636000.]


## Lesson 4: Model Validation

### 1. What is Model Validation?
Model validation is the process of measuring the predictive accuracy of your model. Without it, you cannot reliably test or compare alternative models (like changing features or switching to a Random Forest).

### 2. The Metric: Mean Absolute Error (MAE)
To evaluate a model, we need to summarize thousands of individual predictions into a single, understandable metric. We start with **Mean Absolute Error (MAE)**.

$$\text{Error} = \text{Actual} - \text{Predicted}$$

* **How it works:** We take the absolute value of each prediction error (making all errors positive numbers) and then calculate their average. This is our measure of model quality.
* **In plain English:** "On average, our model's predictions are off by about \$X."

### 3. The Code: Implementing Out-of-Sample Validation
To get a true measure of performance, we use `train_test_split` from `scikit-learn` to break our dataset into two pools:
1. **Training Data (`train_X`, `train_y`):** Used to fit the model.
2. **Validation Data (`val_X`, `val_y`):** Kept hidden during training, used purely to test the model's accuracy on data it hasn't seen before.

In [22]:
from sklearn.metrics import mean_absolute_error

predicted_home_prices = melbourne_model.predict(X)
mean_absolute_error(y, predicted_home_prices)

434.71594577146544

* **In-Sample Error:** When we evaluated the model using the exact same data we used to train it, our error was roughly **\$500**.
* **Out-of-Sample Error:** When we tested it on the hidden validation data, the error jumped to over **\$250,000**!

This massive gap proves that a model can appear nearly perfect on its training data, yet be completely unusable in practice because it memorized specific quirks instead of learning real patterns.

To fix this, we exclude some data from the model-building process, and then use those to test the model's accuracy on data it hasn't seen before. This data is called validation data

- Scikit-learn has a function called train_test_split to break up the data into two pieces
- Use some of that data as training data to fit the model and use the other data as validation data to calculate mean_absolute_error

In [23]:
from sklearn.model_selection import train_test_split

# split data into training and validation data, for both features and target
# The split is based on a random number generator. Supplying a numeric value to
# the random_state argument guarantees we get the same split every time we
# run this script.
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)
# Define model
melbourne_model = DecisionTreeRegressor()
# Fit model
melbourne_model.fit(train_X, train_y)

# get predicted prices on validation data
val_predictions = melbourne_model.predict(val_X)
print(mean_absolute_error(val_y, val_predictions))

259790.5493867011


## Lesson 5: Underfitting and Overfitting

### 1. The Fine-Tuning Dilemma
Every machine learning model faces a natural tension between being too simple or too complex. In Decision Trees, this complexity is controlled heavily by **tree depth** (how many splits the tree makes).



* **Overfitting:** A model matches the training data almost perfectly, but fails to generalize, making terrible predictions on validation/new data. 
    * *Cause:* The tree is too deep. Leaves contain very few houses, meaning the model memorized the specific noise and quirks of the training sample.
* **Underfitting:** A model fails to capture important distinctions and patterns in the data, performing poorly even on the training data.
    * *Cause:* The tree is too shallow. The data isn't divided into distinct enough groups, leaving a wide variety of houses mixed together in the same leaves.

### 2. Finding the Sweet Spot
Our goal is to find the low point of the validation error curve. We can control this trade-off using the `max_leaf_nodes` argument in Scikit-Learn's `DecisionTreeRegressor`, which limits the total number of final leaves our tree can build.

---

### Code Implementation: Optimizing Tree Architecture